# Optuna 優化實驗分析
**實驗：** `optuna_Llama-3.2-1B-Instruct_gsm8k_20260318_031558`

此 notebook 展示各 trial 相較於 baseline 的指標變化（%），並以顏色深淺表示差距大小。

In [12]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

EXP_DIR = Path("optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524")

with open(EXP_DIR / "optimization_results.json") as f:
    results = json.load(f)

with open(EXP_DIR / "experiment_config.json") as f:
    config = json.load(f)

baseline = config["baseline"]
print("Baseline:")
for k, v in baseline.items():
    print(f"  {k}: {v:.4f}")

Baseline:
  accuracy: 0.7180
  latency: 1422.6073
  vram: 6.1952
  emissions: 0.0937


In [13]:
def summarize_config(cfg):
    """將 config dict 轉為簡短描述字串"""
    mode = cfg.get("mode", "unknown")
    parts = []

    if "quant" in cfg:
        q = cfg["quant"]
        method = q.get("method", "?")
        bits = q.get("bits", "?")
        gs = q.get("group_size", "?")
        fmt = q.get("format", "")
        dq = " dq" if q.get("double_quant") else ""
        q_str = f"{method} {bits}bit g{gs}"
        if fmt:
            q_str += f" {fmt}"
        q_str += dq
        parts.append(q_str)

    if "asvd" in cfg:
        a = cfg["asvd"]
        ratio = a.get("ratio", "?")
        alpha = a.get("alpha", "?")
        scaling = a.get("scaling", "?")
        parts.append(f"asvd r{int(ratio*100)} α{int(alpha*10)} {scaling}")

    if "sparse" in cfg:
        s = cfg["sparse"]
        struct = s.get("structure", "?")
        ratio = s.get("ratio", "")
        s_str = f"sparse {struct}"
        if ratio:
            s_str += f" {int(ratio*100)}%"
        parts.append(s_str)

    return " + ".join(parts) if parts else mode


rows = []
for trial in results:
    m = trial["metrics"]
    cfg = trial["config"]
    mode = cfg.get("mode", "unknown")

    if m.get("accuracy") is None or m.get("latency") is None or m.get("vram") is None or m.get("emissions") is None:
        print(f"Trial {trial['iteration']} ({trial['trial_name']}) 沒有完整的 score，跳過")
        continue

    acc_pct   = (m["accuracy"]  - baseline["accuracy"])  / baseline["accuracy"]  * 100
    lat_pct   = (m["latency"]   - baseline["latency"])   / baseline["latency"]   * 100
    vram_pct  = (m["vram"]      - baseline["vram"])      / baseline["vram"]      * 100
    emit_pct  = (m["emissions"] - baseline["emissions"]) / baseline["emissions"] * 100

    rows.append({
        "Trial": trial["iteration"],
        "名稱": trial["trial_name"],
        "Mode": mode,
        "設定": summarize_config(cfg),
        "Score": round(m["score"], 4),
        # 原始值
        "Accuracy": round(m["accuracy"], 4),
        "Latency (s)": round(m["latency"], 1),
        "VRAM (GB)": round(m["vram"], 3),
        "Emissions (g)": round(m["emissions"], 5),
        # 相較 baseline 的 %
        "Δ Accuracy %": round(acc_pct, 1),
        "Δ Latency %": round(lat_pct, 1),
        "Δ VRAM %": round(vram_pct, 1),
        "Δ Emissions %": round(emit_pct, 1),
    })

df = pd.DataFrame(rows).set_index("Trial")
print(f"共 {len(df)} 個 trials")

print("\nBaseline 參考值：")
print(f"  Accuracy : {baseline['accuracy']:.4f}")
print(f"  Latency  : {baseline['latency']:.1f} s")
print(f"  VRAM     : {baseline['vram']:.3f} GB")
print(f"  Emissions: {baseline['emissions']:.5f} g CO₂eq")

Trial 16 (trial_016_asvd_r097_a70) 沒有完整的 score，跳過
共 19 個 trials

Baseline 參考值：
  Accuracy : 0.7180
  Latency  : 1422.6 s
  VRAM     : 6.195 GB
  Emissions: 0.09367 g CO₂eq


In [14]:
def style_table(df):
    display_cols = ["名稱", "Mode", "設定", "Score",
                    "Accuracy", "Latency (s)", "VRAM (GB)", "Emissions (g)",
                    "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]
    d = df[display_cols]

    styled = d.style

    styled = styled.background_gradient(
        subset=["Score"], cmap="RdYlGn", vmin=d["Score"].min(), vmax=d["Score"].max()
    )
    styled = styled.background_gradient(
        subset=["Δ Accuracy %"], cmap="RdYlGn",
        vmin=d["Δ Accuracy %"].min(), vmax=d["Δ Accuracy %"].max()
    )
    for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
        styled = styled.background_gradient(
            subset=[col], cmap="RdYlGn_r",
            vmin=d[col].min(), vmax=d[col].max()
        )

    styled = styled.format({
        "Score": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Latency (s)": "{:.1f}",
        "VRAM (GB)": "{:.3f}",
        "Emissions (g)": "{:.5f}",
        "Δ Accuracy %": "{:+.1f}%",
        "Δ Latency %": "{:+.1f}%",
        "Δ VRAM %": "{:+.1f}%",
        "Δ Emissions %": "{:+.1f}%",
    })

    styled = styled.set_table_styles([
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                      ("font-size", "12px"), ("text-align", "center"),
                                      ("padding", "6px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"),
                                      ("text-align", "center")]},
        {"selector": "tr:hover td", "props": [("filter", "brightness(0.92)")]},
    ])

    return styled

styled_all = style_table(df)
styled_all.set_caption(
    f"全部 30 Trials — Baseline: Acc={baseline['accuracy']:.4f}, "
    f"Lat={baseline['latency']:.1f}s, VRAM={baseline['vram']:.3f}GB, "
    f"Emissions={baseline['emissions']:.5f}g"
)

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,1.1908,0.7060,885.8,4.166,0.05302,-1.7%,-37.7%,-32.8%,-43.4%
2,trial_002_sparse_53pct,sparse_only,sparse unstructured 53%,0.8077,0.2500,401.6,6.196,0.01946,-65.2%,-71.8%,+0.0%,-79.2%
3,trial_003_sparse_33pct,sparse_only,sparse unstructured 33%,1.2585,0.5980,511.1,6.195,0.02480,-16.7%,-64.1%,+0.0%,-73.5%
4,trial_004_sparse_42pct,sparse_only,sparse unstructured 42%,1.2216,0.5360,464.2,6.195,0.02253,-25.3%,-67.4%,+0.0%,-75.9%
5,trial_005_qqq_4bit_g-1,quant_only,qqq 4bit g-1,1.1624,0.4780,727.9,5.288,0.01857,-33.4%,-48.8%,-14.6%,-80.2%
6,trial_006_bnb_8bit,quant_only,bnb 8bit g?,1.1975,0.6760,1590.6,3.594,0.03616,-5.8%,+11.8%,-42.0%,-61.4%
7,trial_007_gptq_2bit_g16_gptq_v2,quant_only,gptq 2bit g16 gptq_v2,-11.4129,0.0000,2169.2,2.064,0.11827,-100.0%,+52.5%,-66.7%,+26.3%
8,trial_008_asvd_r088_a30,asvd_only,asvd r88 α3 fisher,0.0287,0.0660,392.3,5.568,0.01879,-90.8%,-72.4%,-10.1%,-79.9%
9,trial_009_asvd_r095_a50_bnb_4bit,hybrid,bnb 4bit g? dq + asvd r95 α5 fisher,1.0049,0.3500,802.4,2.326,0.02301,-51.3%,-43.6%,-62.5%,-75.4%


In [15]:
from IPython.display import display
pd.set_option("display.max_colwidth", 60)
display(styled_all)

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,1.1908,0.7060,885.8,4.166,0.05302,-1.7%,-37.7%,-32.8%,-43.4%
2,trial_002_sparse_53pct,sparse_only,sparse unstructured 53%,0.8077,0.2500,401.6,6.196,0.01946,-65.2%,-71.8%,+0.0%,-79.2%
3,trial_003_sparse_33pct,sparse_only,sparse unstructured 33%,1.2585,0.5980,511.1,6.195,0.02480,-16.7%,-64.1%,+0.0%,-73.5%
4,trial_004_sparse_42pct,sparse_only,sparse unstructured 42%,1.2216,0.5360,464.2,6.195,0.02253,-25.3%,-67.4%,+0.0%,-75.9%
5,trial_005_qqq_4bit_g-1,quant_only,qqq 4bit g-1,1.1624,0.4780,727.9,5.288,0.01857,-33.4%,-48.8%,-14.6%,-80.2%
6,trial_006_bnb_8bit,quant_only,bnb 8bit g?,1.1975,0.6760,1590.6,3.594,0.03616,-5.8%,+11.8%,-42.0%,-61.4%
7,trial_007_gptq_2bit_g16_gptq_v2,quant_only,gptq 2bit g16 gptq_v2,-11.4129,0.0000,2169.2,2.064,0.11827,-100.0%,+52.5%,-66.7%,+26.3%
8,trial_008_asvd_r088_a30,asvd_only,asvd r88 α3 fisher,0.0287,0.0660,392.3,5.568,0.01879,-90.8%,-72.4%,-10.1%,-79.9%
9,trial_009_asvd_r095_a50_bnb_4bit,hybrid,bnb 4bit g? dq + asvd r95 α5 fisher,1.0049,0.3500,802.4,2.326,0.02301,-51.3%,-43.6%,-62.5%,-75.4%


## Top 10 by Score

In [16]:
top10 = df.sort_values("Score", ascending=False).head(10)
display(style_table(top10).set_caption("Top 10 Trials（依 Score 排序）"))

,名稱,Mode,設定,Score,Accuracy,Latency (s),VRAM (GB),Emissions (g),Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,,,,,
19,trial_019_gptq_4bit_g32_gptq,quant_only,gptq 4bit g32 gptq,1.5177,0.6740,468.4,2.448,0.01614,-6.1%,-67.1%,-60.5%,-82.8%
17,trial_017_awq_4bit_g128,quant_only,awq 4bit g128,1.4568,0.6220,511.9,2.331,0.01686,-13.4%,-64.0%,-62.4%,-82.0%
18,trial_018_bnb_4bit,quant_only,bnb 4bit g?,1.3931,0.6740,735.1,2.495,0.02379,-6.1%,-48.3%,-59.7%,-74.6%
11,trial_011_bnb_4bit,quant_only,bnb 4bit g? dq,1.3508,0.6660,876.8,2.372,0.02663,-7.2%,-38.4%,-61.7%,-71.6%
3,trial_003_sparse_33pct,sparse_only,sparse unstructured 33%,1.2585,0.5980,511.1,6.195,0.02480,-16.7%,-64.1%,+0.0%,-73.5%
4,trial_004_sparse_42pct,sparse_only,sparse unstructured 42%,1.2216,0.5360,464.2,6.195,0.02253,-25.3%,-67.4%,+0.0%,-75.9%
6,trial_006_bnb_8bit,quant_only,bnb 8bit g?,1.1975,0.6760,1590.6,3.594,0.03616,-5.8%,+11.8%,-42.0%,-61.4%
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,1.1908,0.7060,885.8,4.166,0.05302,-1.7%,-37.7%,-32.8%,-43.4%
5,trial_005_qqq_4bit_g-1,quant_only,qqq 4bit g-1,1.1624,0.4780,727.9,5.288,0.01857,-33.4%,-48.8%,-14.6%,-80.2%


## 只看 Delta 欄（排序：Δ Accuracy %）

In [17]:
delta_df = df[["名稱", "Mode", "設定", "Score",
               "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]].sort_values(
    "Δ Accuracy %", ascending=False
)

styled_delta = delta_df.style
styled_delta = styled_delta.background_gradient(subset=["Score"], cmap="RdYlGn")
styled_delta = styled_delta.background_gradient(subset=["Δ Accuracy %"], cmap="RdYlGn")
for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
    styled_delta = styled_delta.background_gradient(subset=[col], cmap="RdYlGn_r")
styled_delta = styled_delta.format({
    "Score": "{:.4f}",
    "Δ Accuracy %": "{:+.1f}%",
    "Δ Latency %": "{:+.1f}%",
    "Δ VRAM %": "{:+.1f}%",
    "Δ Emissions %": "{:+.1f}%",
})
styled_delta = styled_delta.set_table_styles([
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                  ("font-size", "12px"), ("text-align", "center"),
                                  ("padding", "6px 10px")]},
    {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"),
                                  ("text-align", "center")]},
])
styled_delta.set_caption("依 Δ Accuracy % 排序（正值=優於 baseline）")

,名稱,Mode,設定,Score,Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,1.1908,-1.7%,-37.7%,-32.8%,-43.4%
6,trial_006_bnb_8bit,quant_only,bnb 8bit g?,1.1975,-5.8%,+11.8%,-42.0%,-61.4%
19,trial_019_gptq_4bit_g32_gptq,quant_only,gptq 4bit g32 gptq,1.5177,-6.1%,-67.1%,-60.5%,-82.8%
18,trial_018_bnb_4bit,quant_only,bnb 4bit g?,1.3931,-6.1%,-48.3%,-59.7%,-74.6%
11,trial_011_bnb_4bit,quant_only,bnb 4bit g? dq,1.3508,-7.2%,-38.4%,-61.7%,-71.6%
17,trial_017_awq_4bit_g128,quant_only,awq 4bit g128,1.4568,-13.4%,-64.0%,-62.4%,-82.0%
3,trial_003_sparse_33pct,sparse_only,sparse unstructured 33%,1.2585,-16.7%,-64.1%,+0.0%,-73.5%
4,trial_004_sparse_42pct,sparse_only,sparse unstructured 42%,1.2216,-25.3%,-67.4%,+0.0%,-75.9%
5,trial_005_qqq_4bit_g-1,quant_only,qqq 4bit g-1,1.1624,-33.4%,-48.8%,-14.6%,-80.2%


In [18]:
display(styled_delta)

,名稱,Mode,設定,Score,Δ Accuracy %,Δ Latency %,Δ VRAM %,Δ Emissions %
Trial,,,,,,,,
1,trial_001_gptq_8bit_g16_gptq,quant_only,gptq 8bit g16 gptq,1.1908,-1.7%,-37.7%,-32.8%,-43.4%
6,trial_006_bnb_8bit,quant_only,bnb 8bit g?,1.1975,-5.8%,+11.8%,-42.0%,-61.4%
19,trial_019_gptq_4bit_g32_gptq,quant_only,gptq 4bit g32 gptq,1.5177,-6.1%,-67.1%,-60.5%,-82.8%
18,trial_018_bnb_4bit,quant_only,bnb 4bit g?,1.3931,-6.1%,-48.3%,-59.7%,-74.6%
11,trial_011_bnb_4bit,quant_only,bnb 4bit g? dq,1.3508,-7.2%,-38.4%,-61.7%,-71.6%
17,trial_017_awq_4bit_g128,quant_only,awq 4bit g128,1.4568,-13.4%,-64.0%,-62.4%,-82.0%
3,trial_003_sparse_33pct,sparse_only,sparse unstructured 33%,1.2585,-16.7%,-64.1%,+0.0%,-73.5%
4,trial_004_sparse_42pct,sparse_only,sparse unstructured 42%,1.2216,-25.3%,-67.4%,+0.0%,-75.9%
5,trial_005_qqq_4bit_g-1,quant_only,qqq 4bit g-1,1.1624,-33.4%,-48.8%,-14.6%,-80.2%


## 統計摘要

In [19]:
summary = df.groupby("Mode")[["Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %", "Score"]].agg(["mean", "max", "min"]).round(1)
summary.style.background_gradient(cmap="coolwarm", axis=0).set_caption("各 Mode 統計摘要")

In [20]:
# 找出各指標最佳 trial
best_acc  = df.loc[df["Δ Accuracy %"].idxmax()]
best_lat  = df.loc[df["Δ Latency %"].idxmin()]
best_vram = df.loc[df["Δ VRAM %"].idxmin()]
best_emit = df.loc[df["Δ Emissions %"].idxmin()]
best_score = df.loc[df["Score"].idxmax()]

print("=== 各指標最佳 Trial ===")
print(f"最高 Accuracy 改善 : Trial {best_acc.name:>3} | {best_acc['名稱']} | Δ Acc={best_acc['Δ Accuracy %']:+.1f}%")
print(f"最低 Latency 增幅  : Trial {best_lat.name:>3} | {best_lat['名稱']} | Δ Lat={best_lat['Δ Latency %']:+.1f}%")
print(f"最低 VRAM 增幅     : Trial {best_vram.name:>3} | {best_vram['名稱']} | Δ VRAM={best_vram['Δ VRAM %']:+.1f}%")
print(f"最低 Emissions 增幅: Trial {best_emit.name:>3} | {best_emit['名稱']} | Δ Emit={best_emit['Δ Emissions %']:+.1f}%")
print(f"最高 Score         : Trial {best_score.name:>3} | {best_score['名稱']} | Score={best_score['Score']:.4f}")

=== 各指標最佳 Trial ===
最高 Accuracy 改善 : Trial   1 | trial_001_gptq_8bit_g16_gptq | Δ Acc=-1.7%
最低 Latency 增幅  : Trial  14 | trial_014_sparse_4x8 | Δ Lat=-77.0%
最低 VRAM 增幅     : Trial  10 | trial_010_gptq_2bit_g32_gptq | Δ VRAM=-70.3%
最低 Emissions 增幅: Trial  14 | trial_014_sparse_4x8 | Δ Emit=-82.8%
最高 Score         : Trial  19 | trial_019_gptq_4bit_g32_gptq | Score=1.5177


## 重新計算 Score（自訂 Weight）

調整下方 `WEIGHTS` 即可重新計算每個 trial 的分數，並比較排名變化。

公式：
```
norm_acc  = accuracy  / baseline_accuracy
norm_lat  = baseline_latency  / latency
norm_vram = baseline_vram  / vram
norm_emit = baseline_emissions / emissions

score = 1.0 + w_acc*log(norm_acc) + w_lat*log(norm_lat) + w_vram*log(norm_vram) + w_emit*log(norm_emit)
```


In [22]:
# ── 載入所有 3B 實驗的有效 trials ──
ALL_3B_DIRS = sorted(Path(".").glob("optuna_Llama-3.2-3B*"))

all_results = []
all_baseline = None

for exp_dir in ALL_3B_DIRS:
    if not (exp_dir / "optimization_results.json").exists() or not (exp_dir / "experiment_config.json").exists():
        print(f"警告：實驗目錄 {exp_dir} 缺少必要的 JSON 檔案，跳過")
        continue
    with open(exp_dir / "optimization_results.json") as f:
        exp_results = json.load(f)
    with open(exp_dir / "experiment_config.json") as f:
        exp_config = json.load(f)
    if all_baseline is None:
        all_baseline = exp_config["baseline"]
    for trial in exp_results:
        trial["_exp"] = exp_dir.name          # 標記來源實驗
        trial["_id"]  = len(all_results)      # 全域唯一 ID
        all_results.append(trial)

# 建立 all_df（只含有完整 score 的 trials）
b = all_baseline
all_rows = []
for trial in all_results:
    m = trial["metrics"]
    if any(m.get(k) is None for k in ["accuracy", "latency", "vram", "emissions", "score"]):
        continue
    all_rows.append({
        "_id":          trial["_id"],
        "Trial":        trial["iteration"],
        "實驗":         trial["_exp"],
        "名稱":         trial["trial_name"],
        "Mode":         trial["config"].get("mode", "unknown"),
        "設定":         summarize_config(trial["config"]),
        "Score":        round(m["score"], 4),
        "Accuracy":     round(m["accuracy"], 4),
        "Latency (s)":  round(m["latency"], 1),
        "VRAM (GB)":    round(m["vram"], 3),
        "Emissions (g)":round(m["emissions"], 5),
        "Δ Accuracy %": round((m["accuracy"]  - b["accuracy"])  / b["accuracy"]  * 100, 1),
        "Δ Latency %":  round((m["latency"]   - b["latency"])   / b["latency"]   * 100, 1),
        "Δ VRAM %":     round((m["vram"]       - b["vram"])      / b["vram"]      * 100, 1),
        "Δ Emissions %":round((m["emissions"] - b["emissions"]) / b["emissions"] * 100, 1),
    })

all_df = pd.DataFrame(all_rows).set_index("_id")

print(f"讀取 3B 實驗目錄：{len(ALL_3B_DIRS)} 個")
for d in ALL_3B_DIRS:
    cnt = sum(1 for r in all_rows if r["實驗"] == d.name)
    print(f"  {d.name}: {cnt} 個有效 trials")
print(f"\n合計有效 trials: {len(all_df)}")
print(f"Baseline: acc={b['accuracy']:.4f}, lat={b['latency']:.1f}s, "
      f"vram={b['vram']:.3f}GB, emit={b['emissions']:.5f}g")


警告：實驗目錄 optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_130800 缺少必要的 JSON 檔案，跳過
讀取 3B 實驗目錄：4 個
  optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_122557: 11 個有效 trials
  optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_183922: 8 個有效 trials
  optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524: 19 個有效 trials
  optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_130800: 0 個有效 trials

合計有效 trials: 38
Baseline: acc=0.7180, lat=1422.6s, vram=6.195GB, emit=0.09367g


In [23]:
import math
import ipywidgets as widgets
from IPython.display import display

# ── 預設 weight ──
WEIGHTS = {"acc": 1.0, "lat": 1.0, "vram": 1.0, "emit": 1.0}


def recompute_scores(weights, baseline_metrics, source_results=None):
    """依給定 weights 重算所有 trials 的 score，回傳以 _id 為 index 的 DataFrame"""
    if source_results is None:
        source_results = all_results

    base_acc  = baseline_metrics.get("accuracy",   1e-6)
    base_lat  = baseline_metrics.get("latency",    1e-6)
    base_vram = baseline_metrics.get("vram",       1e-6)
    base_emit = baseline_metrics.get("emissions",  1e-6)

    rows = []
    for trial in source_results:
        m = trial["metrics"]
        if any(m.get(k) is None for k in ["accuracy", "latency", "vram", "emissions"]):
            continue

        norm_acc  = m["accuracy"] / (base_acc  + 1e-6)
        norm_lat  = base_lat      / (m["latency"]  + 1e-6)
        norm_vram = base_vram     / (m["vram"]     + 1e-6)
        norm_emit = base_emit     / (m["emissions"]+ 1e-6)

        score = 1.0 + (
            weights.get("acc",  0.0) * math.log(norm_acc  + 1e-9) +
            weights.get("lat",  0.0) * math.log(norm_lat  + 1e-9) +
            weights.get("vram", 0.0) * math.log(norm_vram + 1e-9) +
            weights.get("emit", 0.0) * math.log(norm_emit + 1e-9)
        )

        orig_score = m.get("score")
        rows.append({
            "_id":       trial["_id"],
            "Trial":     trial["iteration"],
            "實驗":      trial["_exp"],
            "名稱":      trial["trial_name"],
            "設定":      summarize_config(trial["config"]),
            "New Score": round(score, 4),
            "Orig Score":round(orig_score, 4) if orig_score is not None else None,
            "Δ Score":   round(score - orig_score, 4) if orig_score is not None else None,
            "Accuracy":  round(m["accuracy"], 4),
            "Δ Accuracy %":  round((m["accuracy"]  - base_acc)  / base_acc  * 100, 1),
            "Δ Latency %":   round((m["latency"]   - base_lat)  / base_lat  * 100, 1),
            "Δ VRAM %":      round((m["vram"]       - base_vram) / base_vram * 100, 1),
            "Δ Emissions %": round((m["emissions"] - base_emit) / base_emit * 100, 1),
        })

    new_df = pd.DataFrame(rows).set_index("_id").sort_values("New Score", ascending=False)
    orig_rank = all_df["Score"].rank(ascending=False).astype(int)
    new_rank  = new_df["New Score"].rank(ascending=False).astype(int)
    new_df["New Rank"]  = new_rank
    new_df["Orig Rank"] = orig_rank.reindex(new_df.index)
    new_df["Rank Δ"]    = new_df["Orig Rank"] - new_df["New Rank"]
    return new_df


def show_recomputed(w_acc, w_lat, w_vram, w_emit):
    weights = {"acc": w_acc, "lat": w_lat, "vram": w_vram, "emit": w_emit}
    new_df = recompute_scores(weights, all_baseline)

    cols = ["實驗", "名稱", "設定", "New Score", "Orig Score", "Δ Score",
            "New Rank", "Orig Rank", "Rank Δ",
            "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]
    styled = new_df[cols].style
    styled = styled.background_gradient(subset=["New Score"], cmap="RdYlGn")
    styled = styled.background_gradient(subset=["Rank Δ"], cmap="RdYlGn",
                                        vmin=-len(new_df), vmax=len(new_df))
    styled = styled.background_gradient(subset=["Δ Accuracy %"], cmap="RdYlGn")
    for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
        styled = styled.background_gradient(subset=[col], cmap="RdYlGn_r")
    styled = styled.format({
        "New Score": "{:.4f}", "Orig Score": "{:.4f}", "Δ Score": "{:+.4f}",
        "Δ Accuracy %": "{:+.1f}%", "Δ Latency %": "{:+.1f}%",
        "Δ VRAM %": "{:+.1f}%", "Δ Emissions %": "{:+.1f}%",
    }).set_table_styles([
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                      ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"), ("text-align", "center")]},
    ])
    top1 = new_df.iloc[0]
    print(f"Weights → acc={w_acc:.2f}  lat={w_lat:.2f}  vram={w_vram:.2f}  emit={w_emit:.2f}  "
          f"| 共 {len(new_df)} trials（來自 {len(ALL_3B_DIRS)} 個 3B 實驗）")
    print(f"最高分: {top1['名稱']} [{top1['實驗'][-15:]}]  Score={top1['New Score']:.4f}")
    display(styled)


style = {"description_width": "80px"}
w_acc  = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_acc",  style=style, continuous_update=False)
w_lat  = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_lat",  style=style, continuous_update=False)
w_vram = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_vram", style=style, continuous_update=False)
w_emit = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_emit", style=style, continuous_update=False)

out = widgets.interactive_output(show_recomputed,
    {"w_acc": w_acc, "w_lat": w_lat, "w_vram": w_vram, "w_emit": w_emit})
display(widgets.VBox([w_acc, w_lat, w_vram, w_emit]), out)


Output()

### 多組 Weight 方案比較

一次比較不同偏好（accuracy優先 / 節能優先 / 均等 / 自訂）下的排名。

In [24]:
WEIGHT_SCENARIOS = {
    "均等 (1:1:1:1)":           {"acc": 1.0, "lat": 1.0, "vram": 1.0, "emit": 1.0},
    "Accuracy 優先 (3:1:1:1)":  {"acc": 3.0, "lat": 1.0, "vram": 1.0, "emit": 1.0},
    "節能優先 (1:1:2:3)":       {"acc": 1.0, "lat": 1.0, "vram": 2.0, "emit": 3.0},
    "速度優先 (1:3:1:1)":       {"acc": 1.0, "lat": 3.0, "vram": 1.0, "emit": 1.0},
    "忽略 Emissions (1:1:1:0)": {"acc": 1.0, "lat": 1.0, "vram": 1.0, "emit": 0.0},
    # "我的方案":               {"acc": 2.0, "lat": 0.5, "vram": 1.5, "emit": 0.5},
}

rank_data = {}
for name, w in WEIGHT_SCENARIOS.items():
    scored = recompute_scores(w, all_baseline)
    rank_data[name] = scored["New Score"].rename(name)

rank_compare = pd.DataFrame(rank_data)
rank_compare.index.name = "_id"

rank_order = rank_compare.rank(ascending=False).astype(int)
rank_order.insert(0, "實驗", all_df["實驗"].reindex(rank_order.index))
rank_order.insert(1, "名稱", all_df["名稱"].reindex(rank_order.index))
rank_order = rank_order.sort_values(list(WEIGHT_SCENARIOS.keys())[0])

print("各方案排名（數字越小越好）")
styled_rank = rank_order.style.background_gradient(
    subset=list(WEIGHT_SCENARIOS.keys()), cmap="RdYlGn_r",
    vmin=1, vmax=len(rank_order)
).set_table_styles([
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                  ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 10px")]},
    {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"), ("text-align", "center")]},
]).set_caption(f"各 Weight 方案排名比較（共 {len(rank_order)} trials，來自 {len(ALL_3B_DIRS)} 個 3B 實驗）")
display(styled_rank)

print("\n各方案實際 Score 數值")
score_compare = rank_compare.copy()
score_compare.insert(0, "實驗", all_df["實驗"].reindex(score_compare.index))
score_compare.insert(1, "名稱", all_df["名稱"].reindex(score_compare.index))
score_compare = score_compare.sort_values(list(WEIGHT_SCENARIOS.keys())[0], ascending=False)
styled_score = score_compare.style.background_gradient(
    subset=list(WEIGHT_SCENARIOS.keys()), cmap="RdYlGn"
).format({k: "{:.4f}" for k in WEIGHT_SCENARIOS}).set_table_styles([
    {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                  ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 10px")]},
    {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"), ("text-align", "center")]},
]).set_caption("各 Weight 方案 Score 數值比較")
display(styled_score)


各方案排名（數字越小越好）


,實驗,名稱,均等 (1:1:1:1),Accuracy 優先 (3:1:1:1),節能優先 (1:1:2:3),速度優先 (1:3:1:1),忽略 Emissions (1:1:1:0)
_id,,,,,,,
23,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_183922,trial_011_bnb_4bit,1,35,1,1,1
42,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_019_gptq_4bit_g32_gptq,2,1,2,2,2
10,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_122557,trial_011_awq_4bit_g32,3,2,3,3,3
40,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_017_awq_4bit_g128,4,3,4,4,4
41,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_018_bnb_4bit,5,4,5,15,5
34,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_011_bnb_4bit,6,5,6,18,6
16,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_183922,trial_004_sparse_42pct,7,12,13,6,17
27,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_004_sparse_42pct,8,10,15,8,14
32,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_009_asvd_r095_a50_bnb_4bit,9,20,7,21,13



各方案實際 Score 數值


,實驗,名稱,均等 (1:1:1:1),Accuracy 優先 (3:1:1:1),節能優先 (1:1:2:3),速度優先 (1:3:1:1),忽略 Emissions (1:1:1:0)
_id,,,,,,,
23,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_183922,trial_011_bnb_4bit,28.4393,-13.0072,66.9736,70.5908,16.9918
42,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_019_gptq_4bit_g32_gptq,4.7345,4.6080,9.1797,6.9563,2.9762
10,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_122557,trial_011_awq_4bit_g32,4.6848,4.5404,9.1185,6.8532,2.9239
40,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_017_awq_4bit_g128,4.5708,4.2837,8.9775,6.6151,2.8562
41,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_018_bnb_4bit,3.8770,3.7505,7.5275,5.1973,2.5065
34,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_011_bnb_4bit,3.6264,3.4761,7.1019,4.5943,2.3686
16,optuna_Llama-3.2-3B-Instruct_gsm8k_20260319_183922,trial_004_sparse_42pct,3.2545,2.4904,6.1952,5.5869,1.7841
27,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_004_sparse_42pct,3.2526,2.6679,6.1025,5.4926,1.8277
32,optuna_Llama-3.2-3B-Instruct_gsm8k_20260320_012524,trial_009_asvd_r095_a50_bnb_4bit,3.2378,1.8007,7.0254,4.3831,1.8338


---

## Penalty 機制

對 accuracy 掉太多的模型額外扣分：

$$\text{Penalty} = a \cdot \max(0,\; (Acc_{base} - t) - Acc)$$

- **$t$（tolerance）**：允許的 accuracy 下降容忍量（絕對值）。掉幅 $\leq t$ 免罰
- **$a$（amplifier）**：超過容忍後每單位掉幅的懲罰倍率

$$\text{Final Score} = \text{Score}(weights) - \text{Penalty}$$


In [25]:
def apply_penalty(base_score, acc, base_acc, t, a):
    """penalty = a * max(0, (base_acc - t) - acc)"""
    penalty = a * max(0.0, (base_acc - t) - acc)
    return base_score - penalty, penalty


def show_with_penalty(w_acc, w_lat, w_vram, w_emit, pen_t, pen_a):
    weights = {"acc": w_acc, "lat": w_lat, "vram": w_vram, "emit": w_emit}
    base_acc = all_baseline["accuracy"]

    scored_df = recompute_scores(weights, all_baseline)

    rows = []
    for idx, row in scored_df.iterrows():
        acc    = all_df.loc[idx, "Accuracy"]
        base_s = row["New Score"]
        final_s, pen = apply_penalty(base_s, acc, base_acc, pen_t, pen_a)
        rows.append({
            "_id":          idx,
            "實驗":         row["實驗"],
            "名稱":         row["名稱"],
            "設定":         row["設定"],
            "Weight Score": round(base_s, 4),
            "Penalty":      round(pen, 4),
            "Final Score":  round(final_s, 4),
            "Accuracy":     round(acc, 4),
            "Δ Accuracy %": row["Δ Accuracy %"],
            "Δ Latency %":  row["Δ Latency %"],
            "Δ VRAM %":     row["Δ VRAM %"],
            "Δ Emissions %":row["Δ Emissions %"],
        })

    pen_df = pd.DataFrame(rows).set_index("_id").sort_values("Final Score", ascending=False)

    orig_rank  = all_df["Score"].rank(ascending=False).astype(int)
    final_rank = pen_df["Final Score"].rank(ascending=False).astype(int)
    pen_df["Final Rank"] = final_rank
    pen_df["Orig Rank"]  = orig_rank.reindex(pen_df.index)
    pen_df["Rank Δ"]     = pen_df["Orig Rank"] - pen_df["Final Rank"]

    cols = ["實驗", "名稱", "設定", "Final Score", "Weight Score", "Penalty",
            "Final Rank", "Orig Rank", "Rank Δ",
            "Accuracy", "Δ Accuracy %", "Δ Latency %", "Δ VRAM %", "Δ Emissions %"]
    styled = pen_df[cols].style
    styled = styled.background_gradient(subset=["Final Score"], cmap="RdYlGn")
    styled = styled.background_gradient(subset=["Penalty"], cmap="RdYlGn_r",
                                        vmin=0, vmax=pen_df["Penalty"].max() or 1)
    styled = styled.background_gradient(subset=["Rank Δ"], cmap="RdYlGn",
                                        vmin=-len(pen_df), vmax=len(pen_df))
    styled = styled.background_gradient(subset=["Δ Accuracy %"], cmap="RdYlGn")
    for col in ["Δ Latency %", "Δ VRAM %", "Δ Emissions %"]:
        styled = styled.background_gradient(subset=[col], cmap="RdYlGn_r")
    styled = styled.format({
        "Final Score": "{:.4f}", "Weight Score": "{:.4f}", "Penalty": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Δ Accuracy %": "{:+.1f}%", "Δ Latency %": "{:+.1f}%",
        "Δ VRAM %": "{:+.1f}%", "Δ Emissions %": "{:+.1f}%",
    }).set_table_styles([
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"),
                                      ("font-size", "12px"), ("text-align", "center"), ("padding", "6px 10px")]},
        {"selector": "td", "props": [("font-size", "11px"), ("padding", "4px 8px"), ("text-align", "center")]},
    ])

    penalized = pen_df[pen_df["Penalty"] > 0]
    top1 = pen_df.iloc[0]
    # print(f"Weights: acc={w_acc:.1f} lat={w_lat:.1f} vram={w_vram:.1f} emit={w_emit:.1f}  |  "
    #       f"Penalty: t={pen_t:.3f}  a={pen_a:.1f}")
    # print(f"容忍門檻: Acc ≥ {base_acc - pen_t:.4f}  "
    #       f"({len(penalized)}/{len(pen_df)} 個 trials 被扣分，共來自 {len(ALL_3B_DIRS)} 個 3B 實驗)")
    # print(f"最高分: {top1['名稱']} [{top1['實驗'][-15:]}]  Final Score={top1['Final Score']:.4f}")
    display(styled)


style = {"description_width": "100px"}
sw_acc  = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_acc",  style=style, continuous_update=False)
sw_lat  = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_lat",  style=style, continuous_update=False)
sw_vram = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_vram", style=style, continuous_update=False)
sw_emit = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="w_emit", style=style, continuous_update=False)

sp_t = widgets.FloatSlider(value=0.01, min=0.0, max=0.20, step=0.005,
                            description="t (tolerance)", style=style,
                            readout_format=".3f", continuous_update=False)
sp_a = widgets.FloatSlider(value=10.0, min=0.0, max=100.0, step=1.0,
                            description="a (amplifier)", style=style,
                            readout_format=".1f", continuous_update=False)

sep = widgets.HTML("<hr style='margin:6px 0'><b>── Penalty ──</b>")

out2 = widgets.interactive_output(show_with_penalty,
    {"w_acc": sw_acc, "w_lat": sw_lat, "w_vram": sw_vram, "w_emit": sw_emit,
     "pen_t": sp_t,   "pen_a": sp_a})
display(widgets.VBox([sw_acc, sw_lat, sw_vram, sw_emit, sep, sp_t, sp_a]), out2)


Output()